# My Learning Journey: Efficient vs Intuitive MHA Implementations

**Date**: February 21, 2026

## What I'm learning

I want to understand the two main ways to implement multi-head attention and see concretely why the weight-split approach is preferred in practice. Both should produce *functionally equivalent* outputs — I'll verify that.

## My questions going in

- Why split weights via `view` + `transpose` instead of just running separate heads?
- Are the outputs truly equivalent, or just approximately the same?
- What does the shape journey look like step by step?

**Attribution**: Concepts from *Build a Large Language Model From Scratch* (Raschka), Ch03 bonus material. Implementation is my own reconstruction.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(123)
print(f"PyTorch: {torch.__version__}")

## Setup: shared input

I'll use a small batch that's easy to inspect: batch_size=2, seq_len=4, embed_dim=4.

In [ ]:
# Tiny dimensions so I can print and inspect everything
BATCH       = 2
SEQ_LEN     = 4
EMBED_DIM   = 4    # d_in and d_out both 4
NUM_HEADS   = 2    # each head sees 2 dims
CTX_LEN     = SEQ_LEN

torch.manual_seed(42)
x = torch.randn(BATCH, SEQ_LEN, EMBED_DIM)
print("Input shape:", x.shape)  # (2, 4, 4)

## Implementation A: Wrapper (stack of single heads)

Each head is a separate `CausalAttention` module. Outputs are concatenated. Simple to follow, but each head runs its own forward pass sequentially.

In [ ]:
class CausalAttention(nn.Module):
    """Single-head causal self-attention."""

    def __init__(self, d_in: int, d_out: int, context_length: int,
                 dropout: float, qkv_bias: bool = False) -> None:
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, n, _ = x.shape
        q = self.W_query(x)
        k = self.W_key(x)
        v = self.W_value(x)
        scores = q @ k.transpose(1, 2)
        scores.masked_fill_(self.mask.bool()[:n, :n], -torch.inf)
        weights = torch.softmax(scores / k.shape[-1]**0.5, dim=-1)
        weights = self.dropout(weights)
        return weights @ v


class MHA_Wrapper(nn.Module):
    """Naive: num_heads separate heads, concatenated output."""

    def __init__(self, d_in: int, d_out: int, context_length: int,
                 dropout: float, num_heads: int) -> None:
        super().__init__()
        assert d_out % num_heads == 0
        head_dim = d_out // num_heads
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, head_dim, context_length, dropout)
             for _ in range(num_heads)]
        )
        self.out_proj = nn.Linear(d_out, d_out)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.out_proj(out)


torch.manual_seed(0)
mha_a = MHA_Wrapper(EMBED_DIM, EMBED_DIM, CTX_LEN, dropout=0.0, num_heads=NUM_HEADS)
out_a = mha_a(x)
print("Wrapper output shape:", out_a.shape)

## Implementation B: Weight-Split (single forward pass)

One big `W_query/key/value` matrix. After projection, I reshape to split the embedding dimension across heads. All heads compute attention simultaneously via batched matrix multiplication.

In [ ]:
class MHA_WeightSplit(nn.Module):
    """Efficient: one QKV projection, reshaped to split across heads."""

    def __init__(self, d_in: int, d_out: int, context_length: int,
                 dropout: float, num_heads: int) -> None:
        super().__init__()
        assert d_out % num_heads == 0
        self.d_out     = d_out
        self.num_heads = num_heads
        self.head_dim  = d_out // num_heads

        self.W_query  = nn.Linear(d_in, d_out)
        self.W_key    = nn.Linear(d_in, d_out)
        self.W_value  = nn.Linear(d_in, d_out)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout  = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def _split(self, proj: torch.Tensor, b: int, n: int) -> torch.Tensor:
        """(b, n, d_out) → (b, num_heads, n, head_dim)"""
        return proj.view(b, n, self.num_heads, self.head_dim).transpose(1, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, n, _ = x.shape
        q = self._split(self.W_query(x), b, n)
        k = self._split(self.W_key(x),   b, n)
        v = self._split(self.W_value(x), b, n)

        scores = q @ k.transpose(2, 3)  # batched over all heads at once
        scores.masked_fill_(self.mask.bool()[:n, :n], -torch.inf)
        weights = torch.softmax(scores / k.shape[-1]**0.5, dim=-1)
        weights = self.dropout(weights)

        # Merge heads: (b, num_heads, n, head_dim) → (b, n, d_out)
        ctx = (weights @ v).transpose(1, 2).contiguous().view(b, n, self.d_out)
        return self.out_proj(ctx)


torch.manual_seed(0)
mha_b = MHA_WeightSplit(EMBED_DIM, EMBED_DIM, CTX_LEN, dropout=0.0, num_heads=NUM_HEADS)
out_b = mha_b(x)
print("Weight-split output shape:", out_b.shape)

## Shape journey (step by step)

I want to trace exactly what happens to tensor dimensions in the weight-split approach.

In [ ]:
# Trace shapes manually using the B implementation's weights
with torch.no_grad():
    b, n, d = x.shape
    num_heads = mha_b.num_heads
    head_dim  = mha_b.head_dim

    projected_q = mha_b.W_query(x)
    print(f"After W_query projection:       {projected_q.shape}")
    # (b, n, d_out) = (2, 4, 4)

    reshaped_q = projected_q.view(b, n, num_heads, head_dim)
    print(f"After view (split heads):        {reshaped_q.shape}")
    # (b, n, num_heads, head_dim) = (2, 4, 2, 2)

    transposed_q = reshaped_q.transpose(1, 2)
    print(f"After transpose (1,2):           {transposed_q.shape}")
    # (b, num_heads, n, head_dim) = (2, 2, 4, 2)

    projected_k = mha_b.W_key(x)
    transposed_k = projected_k.view(b, n, num_heads, head_dim).transpose(1, 2)
    attn_scores = transposed_q @ transposed_k.transpose(2, 3)
    print(f"Attention scores shape:          {attn_scores.shape}")
    # (b, num_heads, n, n) = (2, 2, 4, 4)

## My key insights

1. **Both implementations have the same number of parameters** (when `d_out` is set equivalently). The only structural difference is how the computation is organized.

2. **The `view` trick splits the embedding dimension across heads without copying data.** It's just a different interpretation of the same memory.

3. **`transpose(1, 2)` re-orders dimensions** so PyTorch's batched matrix multiply treats each head independently.

4. **The weight-split approach is preferred in practice** because the single large matrix multiply is more GPU-parallelism-friendly than running sequential head forward passes.

**I still need to explore**: Whether PyTorch's native `nn.MultiheadAttention` uses SDPA (scaled dot-product attention) under the hood for further speedups via FlashAttention.